# Lab 05-01: measuring performance

Lab: `corso/laboratori/lab-05-01-benchmark`

Run the cells in order (Runtime > Run all). It downloads up to 1 GB and takes a few minutes.


## 1. Setup

Download the official llama.cpp binaries (build `b11046`, the one used in the course) and the lab files.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # threads = available cores (more than the physical cores slows things down)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Download the sample model (785 MiB, Apache 2.0) into the Hugging Face cache.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Benchmark (lab 05-01)

`llama bench` measures `pp` (prompt processing) and `tg` (generation) in tokens per second.

In [ ]:
%%bash
M8=$(ls ~/.cache/huggingface/hub/models--ggml-org--Qwen3.5-0.8B-GGUF/snapshots/*/Qwen3.5-0.8B-Q8_0.gguf)
timeout 300 llama bench -m "$M8" -p 64 -n 32 -r 2 -t $T < /dev/null 2>&1 | tail -6

Effect of threads, saved as JSON and plotted.

In [ ]:
%%bash
M8=$(ls ~/.cache/huggingface/hub/models--ggml-org--Qwen3.5-0.8B-GGUF/snapshots/*/Qwen3.5-0.8B-Q8_0.gguf)
cd /tmp
timeout 300 llama bench -m "$M8" -p 32 -n 16 -r 1 -t 1,2 -o json > thread.json
python3 -c "import json; r=json.load(open('thread.json')); [print(x['n_threads'], x['n_prompt'], x['n_gen'], round(x['avg_ts'],2)) for x in r]"

In [ ]:
!pip -q install matplotlib

In [ ]:
%%bash
cd /tmp
python3 $LABS/lab-05-01-benchmark/grafico_bench.py thread.json -o thread.png

In [ ]:
from IPython.display import Image
Image("/tmp/thread.png")